# Bronze - E-commerce Pedidos

## 1. Objetivo

Este notebook realiza a ingestão incremental dos pedidos disponibilizados em micro-lotes no Data Lake para a camada Bronze.

O processo tem como objetivos:

- identificar novos arquivos disponibilizados no diretório `raw/real-time-data`;
- evitar o reprocessamento de micro-lotes já ingeridos por meio de um Manifest;
- identificar novas versões dos pedidos por meio de `id_pedido` e `hash_registro`;
- classificar os registros como `NOVO`, `ATUALIZACAO` ou `DUPLICADO`;
- persistir na camada Bronze apenas versões ainda não processadas;
- manter o histórico das versões ingeridas no Ingestion Control Log;
- particionar fisicamente os dados da Bronze pela data da última atualização;
- executar a ingestão incremental por meio de polling controlado.


##  Autenticação e configurações

###  Bibliotecas utilizadas

As bibliotecas utilizadas no processo são carregadas nesta etapa.


In [0]:
import io
import time
import pandas as pd
import pyarrow.parquet as pq

from pyspark.sql import functions as F

from deltalake import (
    DeltaTable,
    write_deltalake
)

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

###Configuração dos caminhos e parâmetros

As credenciais de acesso são obtidas por meio do notebook de autenticação.

Os dados da camada Bronze serão armazenados fisicamente no Data Lake, enquanto o histórico de ingestão será armazenado em uma estrutura de metadados separada.

In [0]:
%run ./00_autenticação

In [0]:
# Configurações de armazenamento e caminhos do pipeline Bronze.

storage_account = "internshipdatalake"
lake_container = "squad1"

bronze_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"bronze/ecommerce_pedidos"
)

control_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"metadata/ingestion_control_log"
)

manifest_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"metadata/ecommerce_pedidos_manifest"
)

# Ponto inicial do processamento incremental.
# Arquivos anteriores ou iguais a esta baseline
# pertencem ao histórico já existente.
baseline_inicial = (
    "real-time-data/2026/10/01/134202/"
    "ecommerce_pedidos.parquet"
)

# Versão atual do algoritmo de hash utilizado
# para identificar versões dos pedidos.
HASH_VERSION = 2


storage_options = {
    "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
    "AZURE_STORAGE_CLIENT_ID": client_id,
    "AZURE_STORAGE_TENANT_ID": tenant_id,
    "AZURE_STORAGE_CLIENT_SECRET": client_secret
}




## Ingestão de dados em tempo real

A ingestão em tempo real tem como objetivo identificar novos arquivos disponibilizados no Data Lake e incorporá-los à camada Bronze de forma incremental.

O processo inicia-se pelo monitoramento da área destinada aos dados em tempo real. Quando um novo arquivo é identificado, seus registros são lidos, classificados e preparados para persistência na camada Bronze.

Os registros processados também são registrados no Ingestion Control Log, permitindo o controle das versões ingeridas e evitando o processamento duplicado de registros.


In [0]:
# Conexão com o container RAW do Data Lake.
# Esta célula apenas prepara o acesso aos arquivos de origem.


credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

raw_service = DataLakeServiceClient(
    account_url=(
        "https://internshipdatalake.dfs.core.windows.net"
    ),
    credential=credential
)

raw_container = raw_service.get_file_system_client(
    file_system="raw"
)

In [0]:

# Prepara um micro-lote do raw para o processamento da Bronze.
# Esta função apenas lê e transforma os dados; não grava nada.

def preparar_micro_lote(caminho_arquivo):

    # 1. Baixa o arquivo Parquet do RAW.
    arquivo = raw_container.get_file_client(
        caminho_arquivo
    )

    conteudo = arquivo.download_file().readall()

    tabela = pq.read_table(
        io.BytesIO(conteudo)
    )

    df_pandas = tabela.to_pandas()

    # 2. Converte os dados para DataFrame Spark.
    df = spark.createDataFrame(
        df_pandas
    )

    # 3. Padroniza as colunas temporais.
    colunas_timestamp = [
        "dt_pedido",
        "dt_previsao_entrega",
        "dt_ultima_atualizacao_status"
    ]

    for coluna in colunas_timestamp:

        if coluna in df.columns:

            df = df.withColumn(
                coluna,
                F.col(coluna).cast("timestamp")
            )

    # 4. Cria a coluna usada para particionamento da Bronze.
    df = df.withColumn(
        "data_ultima_atualizacao",
        F.col(
            "dt_ultima_atualizacao_status"
        ).cast("date")
    )

    # 5. Gera o hash da versão do pedido.
    #
    # HASH v2:
    # passa a considerar também dt_previsao_entrega.
    df = df.withColumn(
        "hash_registro",
        F.sha2(
            F.concat_ws(
                "||",

                F.coalesce(
                    F.col("id_pedido").cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("id_cliente").cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "id_endereco_entrega"
                    ).cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("dt_pedido").cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "dt_previsao_entrega"
                    ).cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("status_pedido"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("valor_total").cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("valor_frete").cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("metodo_pagamento"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "dt_ultima_atualizacao_status"
                    ).cast("string"),
                    F.lit("")
                )
            ),
            256
        )
    )

    # 6. Identifica qual versão do algoritmo
    # de hash gerou o hash_registro.
    df = df.withColumn(
        "hash_version",
        F.lit(HASH_VERSION).cast("int")
    )

    return df

In [0]:
# Processa um micro-lote completo:
# RAW -> classificação -> Bronze -> Control Log -> Manifest.

def processar_micro_lote(caminho_arquivo):

    print("\n========================================")
    print("Processando:", caminho_arquivo)

    # Evita reprocessar um micro-lote que já está no Manifest.
    if DeltaTable.is_deltatable(
        manifest_path,
        storage_options=storage_options
    ):

        dt_manifest_atual = DeltaTable(
            manifest_path,
            storage_options=storage_options
        )

        arquivos_processados = set(
            dt_manifest_atual
            .to_pandas()["arquivo_origem"]
            .dropna()
            .tolist()
        )

        if caminho_arquivo in arquivos_processados:
            print("Micro-lote já processado. Ignorando.")
            return

    # 1. Prepara o micro-lote.
    df_micro = preparar_micro_lote(
        caminho_arquivo
    )

    qtd_recebida = df_micro.count()

    # 2. Recarrega o Control Log atualizado.
    dt_control_atual = DeltaTable(
        control_path,
        storage_options=storage_options
    )

    df_control_atual = (
        spark.createDataFrame(
            dt_control_atual
            .to_pyarrow_table()
            .to_pandas()
        )
        .select(
            "id_pedido",
            "hash_registro"
        )
        .dropDuplicates()
    )

    # IDs já conhecidos.
    df_ids_processados = (
        df_control_atual
        .select("id_pedido")
        .distinct()
        .withColumn(
            "id_ja_existe",
            F.lit(True)
        )
    )

    # Versões exatas já conhecidas.
    df_versoes_processadas = (
        df_control_atual
        .select(
            "id_pedido",
            "hash_registro"
        )
        .distinct()
        .withColumn(
            "versao_ja_existe",
            F.lit(True)
        )
    )

    # 3. Classifica cada registro.
    df_classificacao = (
        df_micro

        .join(
            df_ids_processados,
            on="id_pedido",
            how="left"
        )

        .join(
            df_versoes_processadas,
            on=[
                "id_pedido",
                "hash_registro"
            ],
            how="left"
        )

        .withColumn(
            "tipo_ocorrencia",
            F.when(
                F.col("versao_ja_existe") == True,
                F.lit("DUPLICADO")
            )
            .when(
                F.col("id_ja_existe") == True,
                F.lit("ATUALIZACAO")
            )
            .otherwise(
                F.lit("NOVO")
            )
        )

        .drop(
            "id_ja_existe",
            "versao_ja_existe"
        )
    )

    # 4. Mantém somente NOVO e ATUALIZACAO.
    df_para_bronze = (
        df_classificacao
        .filter(
            F.col("tipo_ocorrencia").isin(
                "NOVO",
                "ATUALIZACAO"
            )
        )
    )

    qtd_para_bronze = df_para_bronze.count()

    print("Recebidos:", qtd_recebida)
    print("Para Bronze:", qtd_para_bronze)

    # 5. Persiste os dados na Bronze.
    if qtd_para_bronze > 0:

        df_bronze_saida = (
            df_para_bronze
            .drop(
                "hash_registro",
                "tipo_ocorrencia"
            )
        )

        write_deltalake(
            bronze_path,
            df_bronze_saida.toPandas(),
            mode="append",
            partition_by=[
                "data_ultima_atualizacao"
            ],
            schema_mode="merge",
            storage_options=storage_options
        )

        print("Bronze atualizada.")

        # 6. Atualiza o Control Log.
        df_control_novo = (
            df_para_bronze
            .select(
                "id_pedido",
                "hash_registro",
                "dt_ultima_atualizacao_status",
                "tipo_ocorrencia"
            )
            .withColumn(
                "data_ingestao",
                F.current_timestamp()
            )
        )

        write_deltalake(
            control_path,
            df_control_novo.toPandas(),
            mode="append",
            storage_options=storage_options
        )

        print("Control Log atualizado.")

    # 7. Registra o arquivo no Manifest.
    registro_manifest = pd.DataFrame({
        "arquivo_origem": [
            caminho_arquivo
        ],
        "processed_at": [
            pd.Timestamp.now(
                tz="UTC"
            ).tz_localize(None)
        ],
        "qtd_registros": [
            qtd_recebida
        ],
        "qtd_gravados_bronze": [
            qtd_para_bronze
        ]
    })

    write_deltalake(
        manifest_path,
        registro_manifest,
        mode="append",
        storage_options=storage_options
    )

    print("Manifest atualizado.")
    print("Micro-lote concluído ✅")

In [0]:
# Busca os micro-lotes que ainda não foram processados.

def buscar_micro_lotes_pendentes():

    # Lista os arquivos atuais do raw.
    micro_lotes_atual = sorted([
        item.name
        for item in raw_container.get_paths(
            path="real-time-data",
            recursive=True
        )
        if not item.is_directory
        and item.name.endswith("ecommerce_pedidos.parquet")
    ])

    # Consulta o Manifest.
    if DeltaTable.is_deltatable(
        manifest_path,
        storage_options=storage_options
    ):

        dt_manifest_atual = DeltaTable(
            manifest_path,
            storage_options=storage_options
        )

        arquivos_processados = set(
            dt_manifest_atual
            .to_pandas()["arquivo_origem"]
            .dropna()
            .tolist()
        )

    else:
        arquivos_processados = set()

    # Retorna somente arquivos posteriores à baseline
    # que ainda não aparecem no Manifest.
    pendentes = [
        caminho
        for caminho in micro_lotes_atual
        if caminho > baseline_inicial
        and caminho not in arquivos_processados
    ]

    return pendentes

In [0]:
# Executa polling controlado da camada Bronze.
# A cada ciclo verifica novos arquivos, processa e aguarda.

def executar_polling(
    ciclos=10,
    intervalo_segundos=30
):

    for ciclo in range(1, ciclos + 1):

        print("\n========================================")
        print(f"Polling - ciclo {ciclo}/{ciclos}")

        pendentes = buscar_micro_lotes_pendentes()

        print(
            "Micro-lotes pendentes:",
            len(pendentes)
        )

        for caminho in pendentes:
            processar_micro_lote(caminho)

        if ciclo < ciclos:
            print(
                f"Aguardando {intervalo_segundos} segundos..."
            )

            time.sleep(intervalo_segundos)

    print("\nPolling finalizado.")

In [0]:
# Inicia o polling da Bronze por aproximadamente 5 minutos.

executar_polling(
    ciclos=10,
    intervalo_segundos=30
)

In [0]:
# Validação final da ingestão Bronze após o polling.

pendentes_finais = buscar_micro_lotes_pendentes()

dt_manifest_final = DeltaTable(
    manifest_path,
    storage_options=storage_options
)

dt_bronze_final = DeltaTable(
    bronze_path,
    storage_options=storage_options
)

qtd_manifest = len(
    dt_manifest_final.to_pandas()
)

qtd_bronze = (
    dt_bronze_final
    .to_pyarrow_table()
    .num_rows
)

print("=== VALIDAÇÃO FINAL BRONZE ===")
print("Micro-lotes pendentes:", len(pendentes_finais))
print("Arquivos registrados no Manifest:", qtd_manifest)
print("Versão atual da Delta Bronze:", dt_bronze_final.version())
print("Registros atuais na Bronze:", qtd_bronze)

In [0]:
# Verifica a estrutura atual do Ingestion Control Log.

dt_control = DeltaTable(
    control_path,
    storage_options=storage_options
)

tabela_control = dt_control.to_pyarrow_table()

print(tabela_control.schema)

In [0]:
# Diagnóstico antes da migração para o hash v2.
# Esta célula é somente leitura e não altera dados.

dt_bronze_migracao = DeltaTable(
    bronze_path,
    storage_options=storage_options
)

dt_control_migracao = DeltaTable(
    control_path,
    storage_options=storage_options
)

tabela_bronze = (
    dt_bronze_migracao
    .to_pyarrow_table()
)

tabela_control = (
    dt_control_migracao
    .to_pyarrow_table()
)

print("=== BRONZE ===")
print("Registros:", tabela_bronze.num_rows)
print("Versão Delta:", dt_bronze_migracao.version())
print("Colunas:")
print(tabela_bronze.schema)

print("\n=== CONTROL LOG ===")
print("Registros:", tabela_control.num_rows)
print("Versão Delta:", dt_control_migracao.version())
print("Colunas:")
print(tabela_control.schema)

print("\n=== CHECK HASH V2 ===")

tem_previsao = (
    "dt_previsao_entrega"
    in tabela_bronze.column_names
)

tem_hash_version = (
    "hash_version"
    in tabela_control.column_names
)

print(
    "dt_previsao_entrega na Bronze:",
    tem_previsao
)

print(
    "hash_version no Control Log:",
    tem_hash_version
)

if tem_previsao:

    coluna_previsao = tabela_bronze.column(
        "dt_previsao_entrega"
    )

    print(
        "Previsões preenchidas:",
        len(coluna_previsao)
        - coluna_previsao.null_count
    )

    print(
        "Previsões nulas:",
        coluna_previsao.null_count
    )

In [0]:
# Diagnóstico de consistência entre Bronze e Control Log.
# Recalcula o HASH v1 da Bronze para comparar com o histórico atual.
# Esta célula é somente leitura.

df_bronze_check = spark.createDataFrame(
    tabela_bronze.to_pandas()
)

df_control_check = spark.createDataFrame(
    tabela_control.to_pandas()
)

# Recria exatamente o algoritmo HASH v1,
# sem dt_previsao_entrega.
df_bronze_check = df_bronze_check.withColumn(
    "hash_v1",
    F.sha2(
        F.concat_ws(
            "||",

            F.coalesce(
                F.col("id_pedido").cast("string"),
                F.lit("")
            ),

            F.coalesce(
                F.col("id_cliente").cast("string"),
                F.lit("")
            ),

            F.coalesce(
                F.col("id_endereco_entrega").cast("string"),
                F.lit("")
            ),

            F.coalesce(
                F.col("dt_pedido").cast("string"),
                F.lit("")
            ),

            F.coalesce(
                F.col("status_pedido"),
                F.lit("")
            ),

            F.coalesce(
                F.col("valor_total").cast("string"),
                F.lit("")
            ),

            F.coalesce(
                F.col("valor_frete").cast("string"),
                F.lit("")
            ),

            F.coalesce(
                F.col("metodo_pagamento"),
                F.lit("")
            ),

            F.coalesce(
                F.col(
                    "dt_ultima_atualizacao_status"
                ).cast("string"),
                F.lit("")
            )
        ),
        256
    )
)

bronze_versoes_v1 = (
    df_bronze_check
    .select(
        "id_pedido",
        F.col("hash_v1").alias("hash_registro")
    )
    .dropDuplicates()
)

control_versoes_v1 = (
    df_control_check
    .select(
        "id_pedido",
        "hash_registro"
    )
    .dropDuplicates()
)

faltando_no_control = (
    bronze_versoes_v1
    .join(
        control_versoes_v1,
        on=["id_pedido", "hash_registro"],
        how="left_anti"
    )
)

faltando_na_bronze = (
    control_versoes_v1
    .join(
        bronze_versoes_v1,
        on=["id_pedido", "hash_registro"],
        how="left_anti"
    )
)

print("=== CONSISTÊNCIA HASH v1 ===")

print(
    "Versões distintas na Bronze:",
    bronze_versoes_v1.count()
)

print(
    "Versões distintas no Control Log:",
    control_versoes_v1.count()
)

print(
    "Versões da Bronze ausentes no Control Log:",
    faltando_no_control.count()
)

print(
    "Versões do Control Log ausentes na Bronze:",
    faltando_na_bronze.count()
)